# CA Exam Checker AI — Evaluation Supervisor Fine-Tuning (Kaggle GPU)

Use this notebook when **Google Colab has exhausted its free GPU quota**. It runs in Kaggle's notebook environment with a GPU accelerator and exports one ZIP containing the research adapter and evaluation metrics.

**Set up first**
1. Open notebook Settings → Accelerator → **GPU** (P100 or T4 if available).
2. Make sure Internet is enabled so packages and the public base model can be fetched.
3. Run all cells once. If your Kaggle account has no GPU quota or a GPU is unavailable, this provider may also queue/limit access.

This is still a **research-only evaluation supervisor**, not a fully trained CA subject grader. The initial training data is synthetic audit cases. It does not automatically train on retrieved ICAI PDFs, and synthetic metrics are not evidence of real CA grading accuracy. The notebook keeps PDF downloading disabled unless the user has written permission covering the files and AI training/retrieval storage.


In [ ]:
import os, sys, subprocess, json, torch
from pathlib import Path

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(
        "Kaggle GPU is not attached. Open notebook Settings → Accelerator → GPU, then restart and rerun."
    )
print("✅ GPU:", torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())
print("VRAM GiB per first GPU:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("✅ CUDA is available. Avoid turning on GPU for notebooks that only catalogue sources.");


In [ ]:
REPO_DIR = "/kaggle/working/AI-Exam"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "https://github.com/thomasshelby-pixel/AI-Exam.git", REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = f"{REPO_DIR}/research/ca-evaluation-supervisor-ai"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"],
    check=True
)
os.chdir(PROJECT_DIR)
print("✅ Training sources ready:", PROJECT_DIR)


## 1. Source archive — May 2024 through September 2026

This run covers **CA Foundation + CA Intermediate**, from **May 2024 through September 2026 inclusive**. It catalogs MTP/mock tests (Series I and II), matching suggested answers/marking guidance, RTP, actual exam question papers, MCQ keys, applicable study material, amendments, and syllabus/applicability notices.

**Source order:** ICAI/BoS first; if a resource is not found there, secondary archives are listed as unverified fallbacks. The source catalogue is saved in this Kaggle session under `/kaggle/working/CA_Exam_Checker_AI/training_runs`.

**Important rights gate:** public availability is not permission to store ICAI material in a retrieval system. PDF downloads are disabled by default and remain disabled unless written permission/licensing covers the specific files and AI training/retrieval storage. PDF material does not become supervised labels automatically.


In [ ]:
# Batch catalogue: ICAI/BoS first, then secondary archives if a paper is missing
import csv, hashlib, json, re, time, urllib.parse
from datetime import datetime, timezone, date
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import fitz

UTC_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DRIVE_ROOT = Path("/kaggle/working/CA_Exam_Checker_AI/training_runs")
CATALOG_DIR = DRIVE_ROOT / "bos_reference_catalog" / ("May2024_to_Sep2026_" + UTC_STAMP)
CATALOG_DIR.mkdir(parents=True, exist_ok=False)
PDF_DIR = CATALOG_DIR / "reference_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = date(2024, 5, 1)
END_DATE = date(2026, 9, 30)

OFFICIAL_ROOT_PAGES = [
    "https://boslive.icai.org/",
    "https://boslive.icai.org/bos_announcement.php",
    "https://www.icai.org/post/question-papers-foundation-course",
    "https://www.icai.org/post/question-papers-intermediate-course",
    "https://www.icai.org/post/suggested-answer-foundation-course",
    "https://www.icai.org/post/suggested-answer-intermediate-course",
    "https://www.icai.org/post/foundation-course",
    "https://www.icai.org/post/intermediate-course",
    "https://www.icai.org/post/exam-mcq-ques-and-key-may2026",
    "https://boslive.icai.org/index.php/education_content_modelTestPapers.php",
]
# Secondary discovery is fallback-only: ICAI/BoS pages are crawled first.
SECONDARY_ROOT_PAGES = [
    "https://www.caprep.co.in/papers",
    "https://www.caprep.co.in/papers/foundation",
    "https://castudyweb.com/ca-inter-mtp/",
    "https://www.vsijaipur.com/icai-ca-intermediate-papers/",
    "https://cawizard.com/ca-intermediate-question-papers/",
    "https://openmock.in/inter-mtp-past-papers",
    "https://www.jobaaj.com/blog/icai-ca-intermediate-question-paper-2025",
]
OFFICIAL_HOSTS = {"icai.org", "boslive.icai.org", "resource.cdn.icai.org"}
SECONDARY_HOSTS = {
    "caprep.co.in", "castudyweb.com", "vsijaipur.com", "cawizard.com",
    "openmock.in", "jobaaj.com",
}
ALLOWED_HOSTS = OFFICIAL_HOSTS | SECONDARY_HOSTS
DISCOVERY_TERMS = (
    "mock test", "model test", "mtp", "suggested answer", "revision test", "rtp",
    "question paper", "previous year", "mcq", "answer key", "amendment",
    "applicability", "study material", "module", "foundation", "intermediate",
    "september 2026", "may 2026", "january 2026", "september 2025",
    "may 2025", "january 2025", "september 2024", "june 2024", "may 2024",
)
COURSE_TERMS = {
    "CA_FOUNDATION": ("foundation", "c.a. foundation", "ca-foundation"),
    "CA_INTERMEDIATE": ("intermediate", "ca inter", "ca-inter"),
}
TYPE_TERMS = {
    "MTP": ("mock test", "model test", "mtp"),
    "SUGGESTED_ANSWER": ("suggested answer", "suggested answers", "answer sheet", "answer pdf", "solutions"),
    "RTP": ("revision test", "rtp"),
    "PYQ": ("previous year", "question paper", "pyq", "past paper", "past exam"),
    "MCQ_KEY": ("mcq", "answer key", "mcq key"),
    "AMENDMENT": ("amendment", "applicability", "statutory update", "legislative update"),
    "STUDY_MATERIAL": ("study material", "module", "bos module", "study notes"),
}
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (compatible; CAExamCheckerResearch/1.0; source-catalogue)"})

def host_allowed(url):
    try:
        u = urllib.parse.urlparse(url)
        host = (u.hostname or "").lower()
        return u.scheme == "https" and any(host == base or host.endswith("." + base) for base in ALLOWED_HOSTS)
    except Exception:
        return False

def absolutize(base, href):
    url = urllib.parse.urljoin(base, (href or "").strip()).split("#", 1)[0]
    return url if host_allowed(url) else None

def parse_attempt(text):
    x = re.sub(r"[_./-]+", " ", (text or "").lower())
    month_aliases = {
        "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
        "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7, "july": 7,
        "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
        "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
    }
    month_names = {
        1: "JANUARY", 2: "FEBRUARY", 3: "MARCH", 4: "APRIL", 5: "MAY", 6: "JUNE",
        7: "JULY", 8: "AUGUST", 9: "SEPTEMBER", 10: "OCTOBER", 11: "NOVEMBER", 12: "DECEMBER",
    }
    month_regex = r"(january|jan|february|feb|march|mar|april|apr|may|june|jun|july|jul|august|aug|september|sept|sep|october|oct|november|nov|december|dec)"
    patterns = [
        re.compile(r"\b" + month_regex + r"\.?\s*,?\s*(20\d{2})\b", re.I),
        re.compile(r"\b(20\d{2})\s+" + month_regex + r"\b", re.I),
    ]
    matches = []
    for pattern in patterns:
        for match in pattern.finditer(x):
            if pattern is patterns[0]:
                token, year_text = match.group(1).lower(), match.group(2)
            else:
                year_text, token = match.group(1), match.group(2).lower()
            month_num = month_aliases[token]
            matches.append((match.start(), date(int(year_text), month_num, 1), month_names[month_num]))
    if not matches:
        return "ATTEMPT_UNCONFIRMED", None
    _, parsed_date, month_name = min(matches, key=lambda item: item[0])
    return month_name + "_" + str(parsed_date.year), parsed_date

def classify(text, discovered_on):
    x = (text or "").lower()
    course = next((key for key, terms in COURSE_TERMS.items() if any(t in x for t in terms)), "UNKNOWN")
    kind = next((key for key, terms in TYPE_TERMS.items() if any(t in x for t in terms)), "OTHER")
    attempt, attempt_date = parse_attempt(x)
    ser = re.search(r"\bseries\s*[-:]?\s*(i{1,3}|[123])\b", x, re.I)
    series = ("SERIES_" + ({"I":"I","II":"II","III":"III","1":"I","2":"II","3":"III"}[ser.group(1).upper()])) if ser else "SERIES_UNCONFIRMED"
    source_host = (urllib.parse.urlparse(discovered_on).hostname or "").lower()
    source_tier = "OFFICIAL_ICAI_BOS" if any(source_host == h or source_host.endswith("." + h) for h in OFFICIAL_HOSTS) else "SECONDARY_UNVERIFIED"
    if attempt_date:
        date_status = "IN_RANGE" if START_DATE <= attempt_date <= END_DATE else "OUT_OF_RANGE"
    elif kind in {"STUDY_MATERIAL", "AMENDMENT"}:
        date_status = "CURRICULUM_REFERENCE_REVIEW_APPLICABILITY"
    else:
        date_status = "NEEDS_ATTEMPT_REVIEW"
    return course, kind, attempt, series, date_status, source_tier, attempt_date

def anchor_context(a):
    label = " ".join(a.get_text(" ", strip=True).split())
    container = a.find_parent(["tr", "li", "article"])
    if container is None:
        container = a.parent
    contextual = " ".join(container.get_text(" ", strip=True).split()) if container else ""
    prior_heading = a.find_previous(["h1", "h2", "h3", "h4"])
    section_heading = " ".join(prior_heading.get_text(" ", strip=True).split()) if prior_heading else ""
    # Nearest section heading preserves attempt context on archive pages where each paper link just says Download.
    return label, (contextual[:700] if contextual != label else ""), section_heading[:250]

def is_pdf_url(url):
    parsed = urllib.parse.urlparse(url)
    return ".pdf" in parsed.path.lower() or "pdf" in url.lower() or ".zip" in parsed.path.lower()

page_queue = list(dict.fromkeys(OFFICIAL_ROOT_PAGES))
visited_pages, candidates = set(), {}
page_errors = []
# Two passes: official discovery first; secondary archive discovery is only a fallback.
for tier_name, roots in [("OFFICIAL_ICAI_BOS", OFFICIAL_ROOT_PAGES), ("SECONDARY_UNVERIFIED", SECONDARY_ROOT_PAGES)]:
    page_queue = list(dict.fromkeys(roots))
    for depth in range(2):
        next_queue = []
        for page_url in page_queue[:45]:
            if page_url in visited_pages:
                continue
            visited_pages.add(page_url)
            try:
                response = SESSION.get(page_url, timeout=25, allow_redirects=True)
                final_url = response.url
                if not host_allowed(final_url):
                    page_errors.append({"url": page_url, "source_tier": tier_name, "error": "Redirected outside configured source allow-list; skipped."})
                    continue
                response.raise_for_status()
                content_type = response.headers.get("content-type", "").lower()
                if "html" not in content_type and "text" not in content_type:
                    continue
                soup = BeautifulSoup(response.text, "html.parser")
                page_title = soup.title.get_text(" ", strip=True) if soup.title else ""
                headings = " ".join(h.get_text(" ", strip=True) for h in soup.find_all(["h1", "h2", "h3"])[:15])
                for a in soup.find_all("a", href=True):
                    href = absolutize(final_url, a.get("href"))
                    if not href:
                        continue
                    anchor_label, row_context, section_heading = anchor_context(a)
                    label_context = (anchor_label + " | " + section_heading + " | " + row_context)[:1100]
                    context = " ".join([label_context, href, final_url, page_title, headings])
                    if is_pdf_url(href):
                        previous = candidates.get(href, {})
                        prior_text = previous.get("link_text", "")
                        combined_label = " | ".join(dict.fromkeys(x for x in [prior_text, label_context] if x))
                        course, kind, attempt, series, date_status, source_tier, attempt_date = classify(
                            combined_label + " " + href + " " + final_url + " " + page_title,
                            final_url,
                        )
                        # When an answer/question file URL itself is descriptive, use it to avoid mixing paired files.
                        direct_hint = (anchor_label + " " + href).lower()
                        if re.search(r"suggested.?answer|answer.?key|answers?\.pdf|solution", direct_hint):
                            kind = "MCQ_KEY" if "mcq" in direct_hint and "suggested" not in direct_hint else "SUGGESTED_ANSWER"
                        elif re.search(r"question.?paper|questions?\.pdf|qp\.pdf|question.?paper", direct_hint):
                            kind = "MTP" if re.search(r"\bmtp\b|mock.?test|model.?test", (direct_hint + " " + row_context).lower()) else "PYQ"
                        # When the same file is linked by both official and secondary pages, keep official attribution.
                        if previous.get("source_tier") == "OFFICIAL_ICAI_BOS":
                            source_tier = "OFFICIAL_ICAI_BOS"
                        candidates[href] = {
                            "course": course if course != "UNKNOWN" or not previous else previous.get("course", "UNKNOWN"),
                            "type": kind if kind != "OTHER" or not previous else previous.get("type", "OTHER"),
                            "attempt": attempt if attempt != "ATTEMPT_UNCONFIRMED" or not previous else previous.get("attempt", "ATTEMPT_UNCONFIRMED"),
                            "series": series if series != "SERIES_UNCONFIRMED" or not previous else previous.get("series", "SERIES_UNCONFIRMED"),
                            "date_status": date_status if date_status != "NEEDS_ATTEMPT_REVIEW" or not previous else previous.get("date_status", "NEEDS_ATTEMPT_REVIEW"),
                            "source_tier": source_tier,
                            "link_text": combined_label[:900],
                            "url": href,
                            "discovered_on": final_url,
                            "source_page_title": page_title[:300],
                        }
                    elif depth == 0 and any(term in context.lower() for term in DISCOVERY_TERMS):
                        if href not in visited_pages and href not in next_queue:
                            next_queue.append(href)
                time.sleep(0.12)
            except Exception as exc:
                page_errors.append({"url": page_url, "source_tier": tier_name, "error": f"{type(exc).__name__}: {exc}"})
        page_queue = next_queue

candidate_rows = sorted(
    candidates.values(),
    key=lambda x: (x["course"], x["date_status"], x["attempt"], x["type"], x["series"], x["link_text"], x["url"])
)
fieldnames = ["course", "type", "attempt", "series", "date_status", "source_tier", "link_text", "url", "discovered_on", "source_page_title"]
(CATALOG_DIR / "source_catalog.json").write_text(json.dumps(candidate_rows, indent=2, ensure_ascii=False), encoding="utf-8")
with (CATALOG_DIR / "source_catalog.csv").open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows([{k: row.get(k, "") for k in fieldnames} for row in candidate_rows])
(CATALOG_DIR / "discovery_errors.json").write_text(json.dumps(page_errors, indent=2, ensure_ascii=False), encoding="utf-8")

in_range_rows = [r for r in candidate_rows if r["date_status"] == "IN_RANGE"]
official_rows = [r for r in candidate_rows if r["source_tier"] == "OFFICIAL_ICAI_BOS"]
secondary_rows = [r for r in candidate_rows if r["source_tier"] == "SECONDARY_UNVERIFIED"]
print(f"✅ Scanned {len(visited_pages)} unique pages; found {len(candidate_rows)} file links.")
print(f"   Official ICAI/BoS links: {len(official_rows)} | Secondary fallback candidates: {len(secondary_rows)}")
print(f"   Attempt-dated files in May 2024–Sep 2026: {len(in_range_rows)}")
print("Catalogue CSV:", CATALOG_DIR / "source_catalog.csv")
for idx, row in enumerate(candidate_rows, start=1):
    if row["date_status"] in {"IN_RANGE", "CURRICULUM_REFERENCE_REVIEW_APPLICABILITY", "NEEDS_ATTEMPT_REVIEW"}:
        print(f'{idx:04d} | {row["course"]:<17} | {row["type"]:<16} | {row["attempt"]:<22} | {row["series"]:<20} | {row["source_tier"]:<20} | {row["link_text"][:85]} | {row["url"]}')
if page_errors:
    print(f"⚠️ {len(page_errors)} pages could not be read. See discovery_errors.json. Secondary sites can block automated crawling.")

# Confirm rights per source category before bulk PDF storage.
# A YES response is an assertion by the notebook user that written permission/licensing covers
# the specific source files and storage/processing for AI training or retrieval.
print("Permission check: answer only for material whose rights you have actually cleared.")
OFFICIAL_PERMISSION_CONFIRMED = input(
    "Do you have written permission for the selected ICAI/BoS files to be stored/processed for AI training or retrieval? Type YES to confirm: "
).strip().upper() == "YES"
SECONDARY_PERMISSION_CONFIRMED = input(
    "Do you have written permission/licensing for the selected SECONDARY-site files (CAprep/CA Study Web/VSI/CAWizard/OpenMock/Jobaaj) for the same use? Type YES to confirm: "
).strip().upper() == "YES"

MAX_FILES = 2000
MAX_FILE_BYTES = 80 * 1024 * 1024
MAX_TOTAL_BYTES = 8 * 1024 * 1024 * 1024

if not OFFICIAL_PERMISSION_CONFIRMED and not SECONDARY_PERMISSION_CONFIRMED:
    print("\n⏸️ PDF downloads skipped: neither official nor secondary source rights were confirmed.")
    print("The source catalogue is still saved. Re-run this cell and confirm only the source categories covered by your permission.")
else:
    eligible_types = {"MTP", "SUGGESTED_ANSWER", "RTP", "PYQ", "MCQ_KEY", "AMENDMENT", "STUDY_MATERIAL"}
    selected = [
        row for row in candidate_rows
        if row["course"] in {"CA_FOUNDATION", "CA_INTERMEDIATE"}
        and row["type"] in eligible_types
        and row["date_status"] in {"IN_RANGE", "CURRICULUM_REFERENCE_REVIEW_APPLICABILITY"}
        and (
            (row["source_tier"] == "OFFICIAL_ICAI_BOS" and OFFICIAL_PERMISSION_CONFIRMED)
            or (row["source_tier"] == "SECONDARY_UNVERIFIED" and SECONDARY_PERMISSION_CONFIRMED)
        )
    ]
    # Prefer official link copies when the same direct PDF occurs on more than one source page.
    selected.sort(key=lambda x: (x["source_tier"] != "OFFICIAL_ICAI_BOS", x["course"], x["attempt"], x["type"], x["url"]))
    selected = selected[:MAX_FILES]
    total_bytes, file_records = 0, []
    for idx, item in enumerate(selected, start=1):
        url = item["url"]
        rec = {**item, "download_status": "PENDING"}
        try:
            if not host_allowed(url):
                raise ValueError("Source URL outside configured official/secondary allow-list")
            with SESSION.get(url, timeout=45, stream=True, allow_redirects=True) as resp:
                if not host_allowed(resp.url):
                    raise ValueError("Download redirected outside configured source allow-list")
                resp.raise_for_status()
                declared = int(resp.headers.get("content-length") or 0)
                if declared and declared > MAX_FILE_BYTES:
                    raise ValueError(f"File exceeds per-file cap ({declared} bytes)")
                chunks, n = [], 0
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    n += len(chunk)
                    if n > MAX_FILE_BYTES or total_bytes + n > MAX_TOTAL_BYTES:
                        raise ValueError("Download size limit reached")
                    chunks.append(chunk)
                raw = b"".join(chunks)
            if not raw.startswith(b"%PDF"):
                raise ValueError("Response is not a PDF; zip/HTML/drive-view pages need manual review")
            file_name = f'{idx:04d}_{item["course"].lower()}_{item["type"].lower()}_{hashlib.sha256(url.encode()).hexdigest()[:12]}.pdf'
            path = PDF_DIR / file_name
            path.write_bytes(raw)
            total_bytes += len(raw)
            doc = fitz.open(stream=raw, filetype="pdf")
            text = "\n\n".join(f"--- Page {p.number+1} ---\n{p.get_text('text')}" for p in doc).strip()
            text_path = path.with_suffix(".txt")
            if len(text) >= 100:
                text_path.write_text(text, encoding="utf-8")
                extraction_status = "TEXT_EXTRACTED"
            else:
                extraction_status = "OCR_REQUIRED_OR_EMPTY_TEXT"
            rec.update({
                "download_status": "DOWNLOADED",
                "saved_pdf": str(path),
                "sha256": hashlib.sha256(raw).hexdigest(),
                "bytes": len(raw),
                "page_count": len(doc),
                "extracted_text_path": str(text_path) if extraction_status == "TEXT_EXTRACTED" else None,
                "extracted_characters": len(text),
                "extraction_status": extraction_status,
                "training_status": "REFERENCE_ONLY_NOT_A_SUPERVISED_LABEL",
            })
        except Exception as exc:
            rec.update({"download_status": "SKIPPED_OR_FAILED", "error": f"{type(exc).__name__}: {exc}"})
        file_records.append(rec)
        print(f'{idx}/{len(selected)} {rec["download_status"]}: {item["source_tier"]} | {item["course"]} | {item["attempt"]} | {item["type"]} | {item["link_text"][:60]}')
        time.sleep(0.1)
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "date_window": {"start": START_DATE.isoformat(), "end": END_DATE.isoformat()},
        "scope": ["CA_FOUNDATION", "CA_INTERMEDIATE"],
        "types": sorted(eligible_types),
        "source_policy": {
            "official_icai_bos_permission_confirmed_by_user": OFFICIAL_PERMISSION_CONFIRMED,
            "secondary_source_permission_confirmed_by_user": SECONDARY_PERMISSION_CONFIRMED,
            "secondary_sources_are_unverified_fallbacks": True,
            "download_scope": "Only source categories explicitly confirmed for rights-cleared storage/processing are eligible."
        },
        "reference_files_only": True,
        "total_downloaded_bytes": total_bytes,
        "files": file_records,
    }
    (CATALOG_DIR / "reference_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("\n✅ Batch import finished. Materials saved to Drive; nothing has been committed to GitHub.")
    print(json.dumps({"catalog_dir": str(CATALOG_DIR), "downloaded_files": sum(r["download_status"] == "DOWNLOADED" for r in file_records), "bytes": total_bytes}, indent=2))


In [ ]:
# Generate reproducible synthetic examples for train / validation / held-out test.
# This uses no student answer sheets or ICAI solution text.
DATA_DIR = "/kaggle/working/ca-evaluation-supervisor-data"
subprocess.run(
    [sys.executable, f"{PROJECT_DIR}/generate_supervisor_dataset.py",
     "--output-dir", DATA_DIR,
     "--train-per-category", "160",
     "--validation-per-category", "30",
     "--test-per-category", "40",
     "--seed", "20261017"],
    check=True
)
manifest = json.loads(Path(DATA_DIR, "dataset_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
assert manifest["dataset_kind"] == "synthetic_only"
assert manifest["counts"]["train"]["examples"] > manifest["counts"]["test"]["examples"]


## Train an isolated QLoRA adapter

This cell uses the attached Kaggle GPU and writes to a new folder under `/kaggle/working`. If the session ends unexpectedly, Kaggle temporary working files may be lost unless exported/saved as notebook output, so download the final ZIP after training. Keep an eye on the session quota.


In [ ]:
OUTPUT_DIR = "/kaggle/working/evaluation_supervisor_smollm2_360m_v1_20261017"
TRAIN_FILE = "/kaggle/working/ca-evaluation-supervisor-data/train.jsonl"
VALIDATION_FILE = "/kaggle/working/ca-evaluation-supervisor-data/validation.jsonl"
subprocess.run(
    [sys.executable, f"{PROJECT_DIR}/train_supervisor_qlora.py",
     "--train-file", TRAIN_FILE,
     "--validation-file", VALIDATION_FILE,
     "--output-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--epochs", "3",
     "--batch-size", "2",
     "--grad-accum", "8",
     "--learning-rate", "0.0001",
     "--max-seq-length", "1536"],
    check=True
)
if not Path(OUTPUT_DIR, "adapter_config.json").is_file():
    raise RuntimeError("Training returned without adapter_config.json; no package will be created.")
print("✅ Adapter saved:", OUTPUT_DIR)


## Evaluate on held-out synthetic cases

This compares the trained adapter with the base model on a held-out synthetic set. Metrics test the pipeline behavior only; they do **not** mean the model is ready to assign real CA marks.


In [ ]:
EVAL_DIR = "/kaggle/working/evaluation_supervisor_evaluation_v1_20261017"
subprocess.run(
    [sys.executable, f"{PROJECT_DIR}/evaluate_supervisor.py",
     "--test-file", "/kaggle/working/ca-evaluation-supervisor-data/test.jsonl",
     "--adapter-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--output-dir", EVAL_DIR,
     "--limit", "120",
     "--compare-base"],
    check=True
)
metrics_path = Path(EVAL_DIR) / "metrics.json"
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))


In [ ]:
# Package the adapter and evaluation evidence into one downloadable ZIP.
import shutil, zipfile
from IPython.display import FileLink, display

EXPORT_DIR = "/kaggle/working/evaluation_supervisor_smollm2_360m_v1_20261017_EXPORT"
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
for file_path in Path(OUTPUT_DIR).iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, Path(EXPORT_DIR) / file_path.name)
for name in ["metrics.json", "base_metrics.json", "predictions.jsonl", "base_predictions.jsonl"]:
    source = Path(EVAL_DIR) / name
    if source.is_file():
        shutil.copy2(source, Path(EXPORT_DIR) / name)

package_path = "/kaggle/working/evaluation_supervisor_adapter_package_20261017.zip"
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in Path(EXPORT_DIR).iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print("✅ One-file adapter package ready:", package_path)
display(FileLink(package_path))
print("⚠️ Research artifact only. Do not wire into the live website before a human-reviewed real-answer benchmark.");


## Deployment gate — not automatic

The adapter is trained only on synthetic consistency/audit examples. Before any website integration, review:
1. Base vs fine-tuned synthetic metrics, especially false PASS decisions.
2. JSON validity and issue-code behavior.
3. A separate human-reviewed, legally authorized benchmark using real student answers and adjudicated mark allocations.

The adapter must not override deterministic integrity gates or approve a copy when references, answer mapping or handwriting are unresolved.
